# Sets, fields and distributions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/core/03_sets_fields_distributions.ipynb)

The mathematical objects around a model: allowable regions, scalar functions of the state, and probability laws.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. A set is its margin

A `Set` is an allowable region $Z(t)$ written as one vector inequality:

$$z \in Z(t) \iff \mathrm{margin}(z, t) \ge 0 \text{ (every component)}$$

Every tool reads sets through that one method: a planner as a constraint, an evaluator as a
success check, a cost as a penalty. `contains` is derived from it; `sample` and
`bounding_box` are the extras a search or a grid needs. A `BoxSet` has one margin row per
bound: $z - \text{lower}$, then $\text{upper} - z$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from minilink import BoxSet

X = BoxSet([-np.pi, -4.0], [np.pi, 4.0])
z = np.array([1.0, 5.0])

X.margin(z), X.contains(z)

In [ ]:
X.sample(0, 3), X.bounding_box()

A plant declares its own ranges. `plant.state.box` is the state box from the state's bounds
and `plant.input_box` the input box from the input ports' bounds: the usual `X` and `U` of
a planning problem.

In [ ]:
from minilink import Pendulum

plant = Pendulum()
print(plant.state.box)
print(plant.input_box)

## 2. More sets, and intersections

`BallSet(center, radius)` has the one margin $r - \lVert z - c \rVert$; `SingletonSet(point)`
is a single point, the usual start or goal. `A & B` is the `IntersectionSet`: its margin
stacks both, and its bounding box is the tighter of the two. An intersection does not
sample itself; a tool that needs points samples its bounding box and keeps the points
inside.

In [ ]:
from minilink import BallSet, SingletonSet

ball = BallSet([0.0, 0.0], 2.0)
goal = SingletonSet([np.pi, 0.0])
X_ball = X & ball

X_ball.margin(z), X_ball.bounding_box(), goal.contains(np.array([np.pi, 0.0]))

An input set is the same idea for $u \in U(x, t)$: its `margin(u, x, t)` may read the state.
`BoxInputSet.from_bounds` builds a constant box; `BoxInputSet.from_system_inputs(plant)`
reads the plant's port bounds, as `plant.input_box` does.

In [ ]:
from minilink import BoxInputSet

U = BoxInputSet.from_bounds(np.array([-2.0]), np.array([2.0]))
U.margin(np.array([3.0])), U.contains(np.array([1.0]))

## 3. Fields: scalar functions of the state

A `Field` is a scalar function $V(x, u, t)$ with the model's own signature: a Lyapunov
function, a value function, a barrier, a clearance. `QuadraticField(center, P)` is
$V(x) = (x - c)^\top P (x - c)$, with its exact gradient $2 P (x - c)$.

In [ ]:
from minilink.core import QuadraticField

V = QuadraticField(np.zeros(2), np.diag([1.0, 0.25]))
V.value(z), V.gradient(z)

A field turns into the other objects, each conversion named by what it returns:

- `V.as_set(upper=level)` is the sublevel set $\{x : V(x) \le \text{level}\}$; for a
  quadratic field it is an `EllipsoidSet`, which samples and bounds itself;
- `V.as_cost(weight=w)` is the running cost $g = w\,V(x)$, and `V.as_terminal_cost()` the
  terminal cost $h = V(x)$ (see `04_costs`).

The reverse also holds: `MarginField(S)` is the field $V(x) = \min_i \mathrm{margin}_i(x)$
of a set, positive inside and negative outside.

In [ ]:
from minilink.core.fields import MarginField

E = V.as_set(upper=1.0)
clearance = MarginField(X)

type(E).__name__, E.bounding_box(), clearance.value(z), clearance.value(np.zeros(2))

A picture of three of these sets: points drawn uniformly in the box `X`, coloured by
whether they also lie in the ellipsoid `E` intersected with the ball.

In [ ]:
points = X.sample(0, 2000)
E_ball = E & ball
inside = np.array([E_ball.contains(point) for point in points])

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(*points[~inside].T, s=4, c="lightgray", label="X only")
ax.scatter(*points[inside].T, s=4, c="tab:blue", label="E & ball")
ax.set_xlabel("theta")
ax.set_ylabel("dtheta")
ax.legend()
plt.show()

## 4. Distributions: probability, not support

A `Distribution` is a probability law over a vector: `sample(key, n)` draws `n` points
(`key` an integer seed, a NumPy generator or a JAX key), and `mean` and `cov` are its
moments. Its `support` is the set where it puts mass, when that set is bounded. Sets are
support and distributions are probability: a planning problem takes a set for `X0` and a
law for `x_start`, never one for the other.

In [ ]:
from minilink import Gaussian, Uniform
from minilink.core import Particles

start_law = Gaussian(np.zeros(2), std=[0.1, 0.5])
mass_law = Uniform([0.8], [1.2])
cloud = Particles(np.array([[0.0, 0.0], [1.0, 1.0], [2.0, 0.0]]))

start_law.sample(0, 3), start_law.cov, mass_law.support, cloud.mean

## Summary

| Object | Equation | Key methods | Plugs into |
| --- | --- | --- | --- |
| `BoxSet`, `BallSet`, `SingletonSet`, `EllipsoidSet`, `A & B` | $\mathrm{margin}(z, t) \ge 0$ | `margin`, `contains`, `sample`, `bounding_box`, `as_cost`, `with_params` | `X`, `X0`, `Xf` of a problem |
| `BoxInputSet` | $\mathrm{margin}(u, x, t) \ge 0$ | `from_bounds`, `from_system_inputs` | `U` of a problem |
| `QuadraticField`, `MarginField` | $V(x, u, t)$ | `value`, `gradient`, `as_set`, `as_cost`, `as_terminal_cost` | sets and costs |
| `Gaussian`, `Uniform`, `Particles` | a law | `sample`, `mean`, `cov`, `support`, `with_params` | `x_start` and `params` of a problem |

Next, [`04_costs`](04_costs.ipynb): the running and terminal costs that turn a set of
trajectories into a number.